# SocialPulse - Feature Engineering & Model Selection (Week 4)

Visual evaluation of the Week 4 deliverables: the feature-rich dataset, sentiment model selection (in-domain vs deploy-domain), topic model coherence, and engagement/interest summaries. All inputs are produced by the `src/` pipeline (`build_features.py`, `sentiment_model.py --benchmark`, `sentiment_gold_eval.py`).

Headline: the supervised sentiment classifier wins on Twitter (in-domain) but **VADER wins on the YouTube/Instagram gold set** (deploy domain) due to vocabulary domain shift, so VADER is the production scorer. NMF (k=13) is the chosen topic model over LDA.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from wordcloud import WordCloud

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
CLEAN = Path("../data/clean")
EVAL = Path("../data/reports/model_eval")
FEAT = Path("../data/reports/features")
FIG = Path("../data/reports/eda/figures"); FIG.mkdir(parents=True, exist_ok=True)

## 1. Feature-rich dataset

In [ ]:
df = pd.read_csv(CLEAN / "features_dataset.csv")
print("shape:", df.shape)
print("sentiment:", df["sentiment_label"].value_counts().to_dict())
print("engagement tier:", df["engagement_tier"].value_counts().to_dict())
df[["Platform", "Keyword", "clean_text", "lang", "sentiment_label", "dominant_topic_label", "engagement_tier"]].head(6)

## 2. Sentiment model selection - in-domain vs deploy-domain
The key finding: supervised wins on Twitter, VADER wins on the YouTube/Instagram gold set.

In [ ]:
indomain = pd.read_csv(EVAL / "sentiment_benchmark.csv")
gold = pd.read_csv(EVAL / "sentiment_gold_eval.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=indomain.sort_values("macro_f1"), y="model", x="macro_f1", ax=axes[0], color="#4c72b0")
axes[0].set_title("In-domain (Twitter held-out) macro-F1")
sns.barplot(data=gold.sort_values("macro_f1"), y="model", x="macro_f1", ax=axes[1], color="#c44e52")
axes[1].set_title("Deploy-domain (YouTube/Instagram gold) macro-F1")
plt.tight_layout(); plt.savefig(FIG / "sentiment_model_selection.png", dpi=120); plt.show()
print("Production model = highest deploy-domain macro-F1:", gold.sort_values('macro_f1', ascending=False).iloc[0]['model'])

In [ ]:
from textblob import TextBlob
g = pd.read_csv(CLEAN.parent / 'gold' / 'sentiment_gold.csv')
g = g[g['gold_label'].notna()]
labels = ['negative', 'neutral', 'positive']
def _lab(c):
    return 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
pred = g['Content'].fillna('').astype(str).apply(lambda t: _lab(TextBlob(t).sentiment.polarity))
cm = confusion_matrix(g['gold_label'], pred, labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('TextBlob (production) predicted'); plt.ylabel('gold (LLM) label'); plt.title('TextBlob confusion matrix on YT/IG gold')
plt.tight_layout(); plt.savefig(FIG / 'sentiment_confusion_gold.png', dpi=120); plt.show()

## 3. Topic model - coherence sweep (NMF) and top terms

In [ ]:
ks = pd.read_csv(EVAL / "topic_k_selection.csv")
best_k = int(ks.loc[ks["nmf_umass_coherence"].idxmax(), "k"])
ax = ks.plot(x="k", y="nmf_umass_coherence", marker="o", legend=False)
ax.axvline(best_k, color="red", ls="--"); ax.set_ylabel("UMass coherence (relative)")
ax.set_title(f"NMF topic coherence vs k (selected k={best_k})")
plt.tight_layout(); plt.savefig(FIG / "topic_coherence_sweep.png", dpi=120); plt.show()

In [ ]:
terms = pd.read_csv(EVAL / "topic_terms.csv")
top4 = sorted(terms["topic_id"].unique())[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, tid in zip(axes.ravel(), top4):
    sub = terms[terms["topic_id"] == tid]
    freq = {row.term: (11 - row["rank"]) for _, row in sub.iterrows()}
    wc = WordCloud(width=400, height=240, background_color="white").generate_from_frequencies(freq)
    ax.imshow(wc); ax.axis("off"); ax.set_title(f"Topic {tid}")
plt.tight_layout(); plt.savefig(FIG / "topic_wordclouds.png", dpi=120); plt.show()

## 4. Engagement and audience interest

In [ ]:
by_sent = pd.read_csv(FEAT / "engagement_by_sentiment.csv")
by_topic = pd.read_csv(FEAT / "engagement_by_topic.csv").head(8)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=by_sent, x="sentiment_label", y="avg_engagement_pct", ax=axes[0], color="#55a868")
axes[0].set_title("Avg engagement percentile by sentiment")
sns.barplot(data=by_topic, y="dominant_topic_label", x="avg_engagement_pct", ax=axes[1], color="#8172b3")
axes[1].set_title("Top topics by avg engagement percentile")
plt.tight_layout(); plt.savefig(FIG / "engagement_drivers.png", dpi=120); plt.show()

## Key takeaways

- **Sentiment**: supervised TF-IDF wins in-domain (macro-F1 ~0.89) but **VADER wins on the YouTube/Instagram gold set (~0.71 vs ~0.56)** - a clear domain-shift effect, so VADER is the production scorer.
- **Topics**: NMF at k=13 (coherence peak) is far more coherent than LDA; topics map to the audience-interest themes.
- **Engagement**: normalized within platform; engagement compared across sentiment and topic for marketing insight.
- **Storage**: SQLite `post_features` with b-tree indexes + FTS5 full-text search for fast retrieval.
- Full feature documentation in `docs/feature_engineering.md`; all model metrics queryable in the `model_eval` table.